# LAB | API Calling to ChatGPT
Delivered by Carolina, November, 2026

## Summary

**Scenario:** We are working as an AI automation specialist for an e-commerce company. Our team needs to automate the creation of product listings from product images and basic information. Instead of manually writing descriptions, we used ChatGPT's vision capabilities (`gpt-4o`) to generate the listings.

**What we built:** a Python pipeline that:
1. Loads 100 fashion products from the HuggingFace dataset `ashraq/fashion-product-images-small` and saves their images to disk
2. Encodes each image in base64
3. Builds a copywriting prompt from the product data (name, category, colour, gender, season, usage)
4. Sends prompt + image to the OpenAI API and asks for a JSON response (title, description, features, keywords)
5. Runs the pipeline in batch over 5 products and saves the results to `listings.json`

The API key is read from a local `.env` file (listed in `.gitignore`, not committed).

In [1]:
#STEP 1: Setting Up OpenAI API Access

#!pip install python-dotenv

from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
import os #operational system

load_dotenv()  # reads .env and puts its values into the environment

client = OpenAI()

response = client.responses.create(
    model="gpt-4o",
    input="What is machine learning?"
)

print(response.output_text)

Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. It involves training a model on a dataset to recognize patterns and improve its performance over time without being explicitly programmed for specific tasks.

Key aspects of machine learning include:

1. **Data**: The foundation for machine learning models, data is used to train algorithms to make predictions or decisions.

2. **Algorithms**: Methods or procedures that the model uses to learn from data. Common types include supervised learning, unsupervised learning, and reinforcement learning.

3. **Model**: A mathematical representation of the learning process. Once trained, the model can make predictions or decisions.

4. **Training**: The process of feeding data to an algorithm and allowing it to learn underlying patterns.

5. **Evaluation**: Assessing the model’s accura

In [ ]:
# STEP 2: PREPARING THE DATA SET
# Local fallback: create products.json with id, name, category and image_path.
import json
from pathlib import Path
import pandas as pd
from product_listing_helpers import prepare_products

print("Loading product dataset...")
try:
    from datasets import load_dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")
    products_data = list(dataset)
    print(f"Loaded {len(products_data)} products")
except Exception as error:
    print(f"Dataset download failed ({type(error).__name__}); checking products.json.")
    local_path = Path("products.json")
    if not local_path.is_file():
        raise RuntimeError(
            "Dataset unavailable. Create products.json with real local image paths "
            "and rerun this cell. See README.md for the format."
        ) from None
    products_data = json.loads(local_path.read_text(encoding="utf-8"))
    if not isinstance(products_data, list):
        raise ValueError("products.json must contain a JSON array.")

products_df = pd.DataFrame(prepare_products(products_data))
print(f"Dataset prepared: {len(products_df)} products")


In [ ]:
# STEP 3: ENCODING IMAGES FOR API
##Objective: Convert product images to base64 format for API transmission.

import base64
 
def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded
 
# Example usage
sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

In [ ]:
# STEP 4: CREATING THE PRODUCT LISTING PROMPT
##Design an effective prompt for generating product listings.


def create_product_listing_prompt(
    product_name,
    price,
    category,
    additional_info=None,
    tone="casual",
    language_level="simple",
    target_audience="general shoppers"
):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: baseColour, season, or gender-specific information (optional)
    - tone: Writing tone, such as casual or formal
    - language_level: Language complexity, such as simple or technical
    - target_audience: Intended audience for the listing
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Use a {tone} tone.
Use {language_level} language.
Write for {target_audience}.

Product Information:
- Name: {product_name}
- Price: €{price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Analyze the product image and create a compelling product listing.
Return the result as JSON.
"""
    return prompt
 
# Test prompt creation

row = products_df.iloc[0]

test_prompt = create_product_listing_prompt(
    product_name=row["productDisplayName"],
    price=49.99,                      # dataset has no price, so use a sample value
    category=f"{row['masterCategory']} > {row['subCategory']} > {row['articleType']}",
    additional_info=f"Colour: {row['baseColour']}, Gender: {row['gender']}, Season: {row['season']}, Usage: {row['usage']}",
    tone="friendly and casual",
    language_level="simple",
    target_audience="everyday online shoppers"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters

In [ ]:
# STEP 5: Calling the ChatGPT API with Vision
# Shared helper handles image failures, empty responses and invalid listing JSON.
import json
from product_listing_helpers import generate_listing as _generate_listing


def generate_listing(prompt, image_path, model="gpt-4o"):
    return _generate_listing(client, prompt, image_path, model=model)


# Test with one product
listing = generate_listing(test_prompt, row["image_path"])
if listing:
    print(json.dumps(listing, indent=2, ensure_ascii=False))


In [ ]:
#STEP 6: Processing Multiple Products
##Objective: Generate listings for multiple products in batch.

import time

sample = products_df.head(5)  # Process first 5 products for testing
results = []

for index, row in sample.iterrows():
    print(f"Processing {row['id']}...")

    prompt = create_product_listing_prompt(
        product_name=row["productDisplayName"],
        price=49.99,  # Sample price
        category=f"{row['masterCategory']} > {row['subCategory']} > {row['articleType']}",
        additional_info=f"Colour: {row['baseColour']}, Gender: {row['gender']}, Season: {row['season']}, Usage: {row['usage']}"
    )

    listing = generate_listing(prompt, row["image_path"])

    if listing is None:
        status = "failed"
    elif "raw_response" in listing:
        status = "json_error"
    else:
        status = "success"

    results.append({
        "id": row['id'],
        "name": row['productDisplayName'],
        "status": status,
        "listing": listing
    })

    time.sleep(1)

# Save
with open("listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Summary
success_count = sum(1 for r in results if r["status"] == "success")
failure_count = len(results) - success_count
print(f"Successful listings: {success_count}")
print(f"Failed listings: {failure_count}")
